# 从零复现 Swin Transformer：窗口注意力、循环移位与层级特征

本 Notebook 不调用 `timm`、`torchvision.models`、`nn.MultiheadAttention` 或任何现成 Transformer。我们只用基础 PyTorch 手写 patch embedding、window partition/reverse、二维相对位置偏置、shifted-window mask、`SwinBlock.forward`、patch merging 与分类器。

重点不只是“网络能跑”：我们会用可逆性 oracle 检查窗口变换，用概率 oracle 证明循环边界没有泄漏，用相对坐标 oracle 检查 bias 索引，并明确奇数分辨率、非法 shape、梯度和发布制品的边界。全部离线、CPU 单线程；微型条纹任务只证明实现可学习，不代表 ImageNet 泛化。


## 1. 计算图、shape 与复杂度

```text
image [B,C,H,W]
  -> Conv2d(kernel=stride=P)                 -> tokens [B,H/P,W/P,D]
  -> W-MSA block + SW-MSA block              -> [B,h,w,D]
  -> 2×2 PatchMerging                        -> [B,ceil(h/2),ceil(w/2),2D]
  -> W-MSA block -> LayerNorm -> mean -> head -> [B,K]
```

全局注意力对 $N=HW$ 个 token 的注意力矩阵是 $O(N^2)$；窗口大小为 $M$ 时，窗口注意力约为 $O(NM^2)$。shifted window 不改变渐近复杂度，却让相邻 block 中原本分离的窗口交换信息。张量统一采用 `BHWC` 进入 block，因为窗口切分更直观；卷积入口仍是 `NCHW`。


In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)

from copy import deepcopy
from dataclasses import dataclass
from hashlib import sha256
from types import MappingProxyType
import io
import json
import math
import random
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED = 460728
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")

assert DEVICE.type == "cpu"
assert torch.get_num_threads() == 1
assert torch.initial_seed() == SEED
print({"torch": torch.__version__, "device": str(DEVICE), "seed": SEED})


## 2. Patch embedding：卷积就是共享的线性投影

对不重叠的 $P\times P$ patch，`Conv2d(kernel_size=P, stride=P)` 与“展开每个 patch 后乘同一矩阵”等价。这里选择**严格合同**：输入高宽必须能被 patch size 整除，不在模型内部静默裁剪或补零。生产系统若允许任意尺寸，应在预处理 recipe 中明确 pad 值和有效区域 mask。


In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, in_channels=3, embed_dim=24, patch_size=2):
        super().__init__()
        if in_channels <= 0 or embed_dim <= 0 or patch_size <= 0:
            raise ValueError("patch embedding dimensions must be positive")
        self.in_channels = int(in_channels)
        self.embed_dim = int(embed_dim)
        self.patch_size = int(patch_size)
        self.proj = nn.Conv2d(in_channels, embed_dim, patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, images):
        if images.ndim != 4 or images.shape[1] != self.in_channels:
            raise ValueError("expected NCHW with configured channels")
        if not torch.is_floating_point(images) or not torch.isfinite(images).all():
            raise ValueError("images must be finite floating point")
        h, w = images.shape[-2:]
        if h % self.patch_size or w % self.patch_size:
            raise ValueError("height and width must be divisible by patch_size")
        return self.norm(self.proj(images).permute(0, 2, 3, 1))

patcher = PatchEmbedding(3, 12, 2)
patch_x = torch.randn(2, 3, 8, 10, requires_grad=True)
patch_y = patcher(patch_x)
assert patch_y.shape == (2, 4, 5, 12)
patch_y.square().mean().backward()
assert patch_x.grad is not None and torch.isfinite(patch_x.grad).all()
assert float(patch_x.grad.abs().sum()) > 0
try:
    patcher(torch.randn(1, 3, 7, 8))
    raise AssertionError("odd incompatible height must fail")
except ValueError:
    pass


## 3. Window partition/reverse：先证明是双射

`window_partition` 把 `[B,H,W,C]` 重排为 `[B*nW,M*M,C]`；`window_reverse` 必须精确恢复原张量。二者只有 `view/permute`，不应丢值或改变 token 次序。与其只检查 shape，更强的 oracle 是给每个位置唯一编号并逐元素比较往返结果。


In [ ]:
def window_partition(x, window_size):
    if x.ndim != 4 or window_size <= 0:
        raise ValueError("x must be BHWC and window_size positive")
    b, h, w, c = x.shape
    if h % window_size or w % window_size:
        raise ValueError("feature resolution must be divisible by window_size")
    x = x.reshape(b, h // window_size, window_size, w // window_size, window_size, c)
    return x.permute(0, 1, 3, 2, 4, 5).contiguous().reshape(-1, window_size ** 2, c)

def window_reverse(windows, window_size, height, width, batch_size):
    if windows.ndim != 3 or min(window_size, height, width, batch_size) <= 0:
        raise ValueError("invalid window reverse arguments")
    if height % window_size or width % window_size:
        raise ValueError("height/width must be divisible by window_size")
    expected = batch_size * (height // window_size) * (width // window_size)
    if windows.shape[0] != expected or windows.shape[1] != window_size ** 2:
        raise ValueError("window count or token count mismatch")
    c = windows.shape[-1]
    x = windows.reshape(batch_size, height // window_size, width // window_size,
                        window_size, window_size, c)
    return x.permute(0, 1, 3, 2, 4, 5).contiguous().reshape(batch_size, height, width, c)

numbered = torch.arange(2 * 8 * 12 * 3).reshape(2, 8, 12, 3)
numbered_windows = window_partition(numbered, 4)
numbered_back = window_reverse(numbered_windows, 4, 8, 12, 2)
assert numbered_windows.shape == (12, 16, 3)
assert torch.equal(numbered, numbered_back)
try:
    window_partition(torch.zeros(1, 7, 8, 2), 4)
    raise AssertionError("non-divisible feature must fail")
except ValueError:
    pass


## 4. 手写窗口注意力与二维相对位置偏置

每个 head 的注意力为

$$A=\operatorname{softmax}(QK^\top/\sqrt{d_h}+B_{\Delta h,\Delta w}+M).$$

窗口内两个位置的相对位移各落在 `[-M+1,M-1]`，所以 bias table 有 $(2M-1)^2$ 行。索引必须同时编码行差与列差；只按一维距离会错误地把“上方”和“左方”视为同一位置。下面的反向索引恒等式能抓出符号或 stride 写错。


In [ ]:
class WindowAttention(nn.Module):
    def __init__(self, dim, window_size, num_heads):
        super().__init__()
        if dim <= 0 or window_size <= 0 or num_heads <= 0 or dim % num_heads:
            raise ValueError("dim must be divisible by positive num_heads")
        self.dim, self.window_size, self.num_heads = int(dim), int(window_size), int(num_heads)
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, 3 * dim)
        self.proj = nn.Linear(dim, dim)
        table_size = (2 * window_size - 1) ** 2
        self.relative_position_bias_table = nn.Parameter(torch.zeros(table_size, num_heads))
        nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)

        coords = torch.stack(torch.meshgrid(torch.arange(window_size), torch.arange(window_size), indexing="ij"))
        flat = coords.flatten(1)
        relative = flat[:, :, None] - flat[:, None, :]
        relative = relative.permute(1, 2, 0).contiguous()
        relative[:, :, 0] += window_size - 1
        relative[:, :, 1] += window_size - 1
        relative[:, :, 0] *= 2 * window_size - 1
        self.register_buffer("relative_position_index", relative.sum(-1).long())

    def forward(self, windows, attention_mask=None, return_attention=False):
        if windows.ndim != 3 or windows.shape[1] != self.window_size ** 2 or windows.shape[2] != self.dim:
            raise ValueError("windows must be [BnW, M*M, dim]")
        if not torch.isfinite(windows).all():
            raise ValueError("window tokens must be finite")
        bnw, n, _ = windows.shape
        qkv = self.qkv(windows).reshape(bnw, n, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        scores = (q @ k.transpose(-2, -1)) * self.scale
        bias = self.relative_position_bias_table[self.relative_position_index.reshape(-1)]
        bias = bias.reshape(n, n, self.num_heads).permute(2, 0, 1)
        scores = scores + bias.unsqueeze(0)
        if attention_mask is not None:
            if attention_mask.ndim != 3 or attention_mask.shape[0] == 0 or attention_mask.shape[1:] != (n, n):
                raise ValueError("attention_mask must be nonempty [nW, M*M, M*M]")
            if not torch.is_floating_point(attention_mask) or not torch.isfinite(attention_mask).all():
                raise ValueError("attention_mask must be finite floating point")
            if attention_mask.device != scores.device or attention_mask.dtype != scores.dtype:
                raise ValueError("attention_mask must match attention score dtype/device")
            # 0 表示允许；小于等于 -20 的值才足以作为 float32 安全屏蔽。
            allowed = (attention_mask == 0) | (attention_mask <= -20)
            if not bool(allowed.all()):
                raise ValueError("attention_mask values must be 0 or a safe negative blocker <= -20")
            nwin = attention_mask.shape[0]
            if bnw % nwin:
                raise ValueError("batch-window count is incompatible with mask")
            scores = scores.reshape(bnw // nwin, nwin, self.num_heads, n, n)
            scores = scores + attention_mask[None, :, None]
            scores = scores.reshape(bnw, self.num_heads, n, n)
        probs = scores.softmax(dim=-1)
        out = (probs @ v).transpose(1, 2).reshape(bnw, n, self.dim)
        out = self.proj(out)
        return (out, probs) if return_attention else out

attn_probe = WindowAttention(16, 4, 4)
idx = attn_probe.relative_position_index
center = (4 - 1) * (2 * 4 - 1) + (4 - 1)
assert idx.shape == (16, 16)
assert torch.equal(torch.diag(idx), torch.full((16,), center, dtype=torch.long))
assert torch.equal(idx + idx.T, torch.full_like(idx, 2 * center))
assert int(idx.min()) == 0 and int(idx.max()) == (2 * 4 - 1) ** 2 - 1
attn_input = torch.randn(3, 16, 16, requires_grad=True)
attn_output = attn_probe(attn_input)
attn_output.sum().backward()
assert attn_output.shape == attn_input.shape
assert attn_input.grad is not None and torch.isfinite(attn_input.grad).all()
assert attn_probe.scale == 0.5  # head_dim=4 的 1/sqrt(d_h) 数值 oracle
assert int(idx[0, 1]) == 23 and int(idx[0, 4]) == 17

valid_mask46 = torch.zeros(1, 16, 16)
valid_mask46[:, 0, 1] = -100.0
valid_out46, valid_probs46 = attn_probe(attn_input.detach(), valid_mask46, return_attention=True)
assert torch.isfinite(valid_out46).all()
assert float(valid_probs46[:, :, 0, 1].max()) < 1e-40

bad_masks46 = [
    torch.full((1, 16, 16), float("nan")),
    torch.full((1, 16, 16), 0.25),
    torch.zeros(1, 16, 16, dtype=torch.float64),
]
for bad_mask46 in bad_masks46:
    try:
        attn_probe(attn_input.detach(), bad_mask46)
        raise AssertionError("非法 attention mask 未 fail-closed")
    except ValueError as exc:
        assert "attention_mask" in str(exc)


## 5. Shifted-window mask：允许邻窗交流，但禁止环绕边界“穿越”

SW-MSA 先把特征循环左移/上移，再按固定窗口切分。循环移位会把图像最右侧卷到最左侧；这些并非真实邻居，必须用 region mask 阻断。注意：mask **不能**简单按原窗口 ID 分组，否则所有跨窗口连接都会被禁掉，SW-MSA 就退化了。

标准做法把每个轴切为 `[0:-M]、[-M:-s]、[-s:]` 三段，二维组合成九个 region。窗口内不同 region 的 pair 加 `-100`；相同 region 加 0。


In [ ]:
def build_shifted_window_mask(height, width, window_size, shift_size, device=None):
    if min(height, width, window_size) <= 0 or not 0 < shift_size < window_size:
        raise ValueError("require 0 < shift_size < window_size")
    if height % window_size or width % window_size:
        raise ValueError("resolution must be divisible by window_size")
    region = torch.zeros((1, height, width, 1), device=device)
    h_slices = (slice(0, -window_size), slice(-window_size, -shift_size), slice(-shift_size, None))
    w_slices = (slice(0, -window_size), slice(-window_size, -shift_size), slice(-shift_size, None))
    count = 0
    for hs in h_slices:
        for ws in w_slices:
            region[:, hs, ws, :] = count
            count += 1
    region_windows = window_partition(region, window_size).squeeze(-1)
    differences = region_windows[:, :, None] - region_windows[:, None, :]
    return differences.ne(0).to(torch.float32) * -100.0

shift_mask = build_shifted_window_mask(8, 8, 4, 2)
assert shift_mask.shape == (4, 16, 16)
assert set(shift_mask.unique().tolist()) == {-100.0, -0.0}
assert torch.equal(torch.diagonal(shift_mask, dim1=-2, dim2=-1), torch.zeros(4, 16))
assert bool((shift_mask == -100).any())
assert bool(((shift_mask == 0) & ~torch.eye(16, dtype=torch.bool)[None]).any())

# 概率 oracle：被 mask 的 pair 在 softmax 后必须数值上为零，而非“只是较小”。
zero_scores = torch.zeros_like(shift_mask)
masked_probs = (zero_scores + shift_mask).softmax(-1)
assert float(masked_probs[shift_mask == -100].max()) < 1e-40
assert torch.allclose(masked_probs.sum(-1), torch.ones_like(masked_probs.sum(-1)))
try:
    build_shifted_window_mask(7, 8, 4, 2)
    raise AssertionError("illegal shifted resolution must fail")
except ValueError:
    pass


## 6. Swin block：Pre-Norm、残差与 cyclic shift

一个 block 先 `LayerNorm -> (S)W-MSA -> residual`，再 `LayerNorm -> MLP -> residual`。shift block 的顺序是：负向 roll、窗口切分、带 mask 注意力、窗口恢复、正向 roll。`shift_size=0` 的普通窗口 block 不使用 mask。DropPath 为教学简洁省略，不影响核心算子定义。


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, dim)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class SwinBlock(nn.Module):
    def __init__(self, dim, input_resolution, num_heads, window_size=4, shift_size=0, mlp_ratio=2):
        super().__init__()
        h, w = input_resolution
        if h % window_size or w % window_size or not 0 <= shift_size < window_size:
            raise ValueError("resolution/shift is incompatible with window")
        self.dim = int(dim)
        self.input_resolution = (int(h), int(w))
        self.window_size, self.shift_size = int(window_size), int(shift_size)
        self.norm1 = nn.LayerNorm(dim)
        self.attention = WindowAttention(dim, window_size, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = FeedForward(dim, dim * mlp_ratio)
        if shift_size:
            mask = build_shifted_window_mask(h, w, window_size, shift_size)
        else:
            mask = torch.empty(0)
        self.register_buffer("attention_mask", mask)

    def forward(self, x):
        if x.ndim != 4 or tuple(x.shape[1:3]) != self.input_resolution or x.shape[-1] != self.dim:
            raise ValueError("SwinBlock received an incompatible BHWC tensor")
        shortcut = x
        y = self.norm1(x)
        if self.shift_size:
            y = torch.roll(y, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
        windows = window_partition(y, self.window_size)
        mask = self.attention_mask if self.shift_size else None
        windows = self.attention(windows, mask)
        y = window_reverse(windows, self.window_size, *self.input_resolution, x.shape[0])
        if self.shift_size:
            y = torch.roll(y, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        x = shortcut + y
        return x + self.mlp(self.norm2(x))

regular_block = SwinBlock(16, (8, 8), 4, 4, 0)
shifted_block = SwinBlock(16, (8, 8), 4, 4, 2)
block_x = torch.randn(2, 8, 8, 16, requires_grad=True)
block_y = shifted_block(regular_block(block_x))
block_y.mean().backward()
assert block_y.shape == block_x.shape
assert block_x.grad is not None and torch.isfinite(block_x.grad).all()
assert float(block_x.grad.norm()) > 0
assert shifted_block.attention_mask.numel() == 4 * 16 * 16


## 7. Patch merging：显式处理奇数分辨率

层级视觉模型需要降采样。`PatchMerging` 取 `(偶行偶列、奇行偶列、偶行奇列、奇行奇列)` 四组 token，在通道维拼成 `4C`，归一化后投影到 `2C`。

这里与 patch embedding 的严格策略不同：中间特征可能因真实输入长宽而成为奇数，所以明确在**右侧/底部补零**并返回 `ceil(H/2),ceil(W/2)`。这不是悄悄行为：artifact 的 coordinate recipe 会绑定这一约定。


In [ ]:
class PatchMerging(nn.Module):
    def __init__(self, dim):
        super().__init__()
        if dim <= 0:
            raise ValueError("dim must be positive")
        self.dim = int(dim)
        self.norm = nn.LayerNorm(4 * dim)
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)

    def forward(self, x):
        if x.ndim != 4 or x.shape[-1] != self.dim:
            raise ValueError("PatchMerging expects BHWC with configured dim")
        b, h, w, c = x.shape
        if h % 2 or w % 2:
            x = F.pad(x, (0, 0, 0, w % 2, 0, h % 2))
        x00, x10 = x[:, 0::2, 0::2], x[:, 1::2, 0::2]
        x01, x11 = x[:, 0::2, 1::2], x[:, 1::2, 1::2]
        merged = torch.cat([x00, x10, x01, x11], dim=-1)
        return self.reduction(self.norm(merged))

merger = PatchMerging(6)
even_merged = merger(torch.randn(2, 8, 6, 6))
odd_merged = merger(torch.randn(2, 5, 7, 6))
assert even_merged.shape == (2, 4, 3, 12)
assert odd_merged.shape == (2, 3, 4, 12)
try:
    merger(torch.randn(1, 5, 7, 5))
    raise AssertionError("wrong channel count must fail")
except ValueError:
    pass


## 8. 组装 Tiny Swin 分类器

教学模型使用 `16×16` 输入、`2×2` patch、两层 stage。第一 stage 的两个 block 分别使用 W-MSA 与 SW-MSA；合并后通道翻倍，再做一个普通窗口 block。真实 Swin-T 有更多 block、stochastic depth 和精心调参，这里保留核心 forward 数据流。


In [ ]:
class TinySwinClassifier(nn.Module):
    def __init__(self, image_size=16, in_channels=1, num_classes=3, embed_dim=16):
        super().__init__()
        if image_size != 16:
            raise ValueError("this audited teaching configuration binds image_size=16")
        self.image_size, self.in_channels, self.num_classes = image_size, in_channels, num_classes
        self.patch_embed = PatchEmbedding(in_channels, embed_dim, patch_size=2)
        self.stage1 = nn.ModuleList([
            SwinBlock(embed_dim, (8, 8), 4, 4, 0),
            SwinBlock(embed_dim, (8, 8), 4, 4, 2),
        ])
        self.merge = PatchMerging(embed_dim)
        self.stage2 = SwinBlock(2 * embed_dim, (4, 4), 4, 2, 0)
        self.norm = nn.LayerNorm(2 * embed_dim)
        self.head = nn.Linear(2 * embed_dim, num_classes)

    def forward(self, images):
        if images.ndim != 4 or images.shape[1:] != (self.in_channels, self.image_size, self.image_size):
            raise ValueError("classifier expects the published NCHW shape")
        x = self.patch_embed(images)
        for block in self.stage1:
            x = block(x)
        x = self.merge(x)
        x = self.stage2(x)
        return self.head(self.norm(x).mean(dim=(1, 2)))

swin_model = TinySwinClassifier().to(DEVICE)
swin_logits = swin_model(torch.randn(4, 1, 16, 16))
assert swin_logits.shape == (4, 3)
assert torch.isfinite(swin_logits).all()
parameter_count = sum(p.numel() for p in swin_model.parameters())
assert 10_000 < parameter_count < 100_000


## 9. 受控训练：只验证端到端梯度，不冒充视觉泛化

我们合成三类极简单图案：竖条、横条和十字，并固定 train/test seed。目标是检查 patch、attention、merge、head 的梯度能共同降低损失。因为分布是人工设计且样本极少，即使准确率很高，也不能外推到自然图像。


In [ ]:
def make_swin_patterns(count, seed):
    generator = torch.Generator().manual_seed(seed)
    labels = torch.arange(count) % 3
    images = 0.04 * torch.randn(count, 1, 16, 16, generator=generator)
    for i, label in enumerate(labels.tolist()):
        offset = int(torch.randint(5, 11, (1,), generator=generator))
        if label in (0, 2):
            images[i, :, :, offset-1:offset+1] += 1.0
        if label in (1, 2):
            images[i, :, offset-1:offset+1, :] += 1.0
    return images.clamp(0, 1), labels.long()

train_images46, train_labels46 = make_swin_patterns(36, SEED + 1)
test_images46, test_labels46 = make_swin_patterns(18, SEED + 2)
assert not torch.equal(train_images46[:18], test_images46)
assert set(train_labels46.tolist()) == {0, 1, 2}

optimizer = torch.optim.AdamW(swin_model.parameters(), lr=5e-3, weight_decay=1e-4)
loss_trace46 = []
swin_model.train()
for step in range(38):
    optimizer.zero_grad(set_to_none=True)
    loss = F.cross_entropy(swin_model(train_images46), train_labels46)
    loss.backward()
    optimizer.step()
    loss_trace46.append(float(loss.detach()))

swin_model.eval()
with torch.no_grad():
    test_accuracy46 = float((swin_model(test_images46).argmax(-1) == test_labels46).float().mean())
assert loss_trace46[-1] < 0.25 * loss_trace46[0]
assert test_accuracy46 >= 0.94
assert all(math.isfinite(v) for v in loss_trace46)
print({"initial_loss": round(loss_trace46[0], 4), "final_loss": round(loss_trace46[-1], 4),
       "controlled_test_accuracy": test_accuracy46})


## 10. 发布制品：认证完整 package，并返回不可变语义 bundle

攻击者若能整体替换权重、配置并重算 package 内所有 hash，“hash 都一致”仍不能证明这是发布方批准的模型。因此 loader 除内部一致性外，还查询 package **之外**的只读 publisher registry：`artifact_id -> expected package digest`。

canonical state digest 按排序后的参数名绑定 `key + dtype + shape + raw bytes`。整体 digest 还覆盖 architecture、attention/mask、数据 split、像素预处理、patch/merge 坐标与标签语义。loader 会逐项与代码支持的合同交叉校验，不能只依赖“已签名但内部互相矛盾”的 metadata。

返回值是 `PublishedSwin`，而非丢失标签和预处理语义的裸 `nn.Module`。其中 metadata 递归冻结；`predict` 同时执行输入范围和 dtype 合同。


In [ ]:
def state_digest46(state):
    digest = sha256()
    for key in sorted(state):
        tensor = state[key].detach().cpu().contiguous()
        digest.update(key.encode())
        digest.update(str(tensor.dtype).encode())
        digest.update(json.dumps(list(tensor.shape)).encode())
        digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()

def tensor_digest46(*tensors):
    digest = sha256()
    for tensor in tensors:
        value = tensor.detach().cpu().contiguous()
        digest.update(str(value.dtype).encode())
        digest.update(json.dumps(list(value.shape)).encode())
        digest.update(value.numpy().tobytes())
    return digest.hexdigest()

def package_digest46(package):
    payload = {k: package[k] for k in sorted(package) if k != "package_digest"}
    return sha256(json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()).hexdigest()

def deep_freeze46(value):
    if isinstance(value, dict):
        return MappingProxyType({key: deep_freeze46(item) for key, item in value.items()})
    if isinstance(value, list):
        return tuple(deep_freeze46(item) for item in value)
    return value

CONFIG46 = {"image_size": 16, "in_channels": 1, "num_classes": 3, "embed_dim": 16,
            "patch_size": 2, "window_sizes": [4, 2], "shift_size": 2}
SPLIT46 = {"generator": "torch.Generator", "train_seed": SEED + 1, "test_seed": SEED + 2,
           "train_count": 36, "test_count": 18, "selection": "fixed-steps-no-test-selection"}
PREPROCESS46 = {"range": [0.0, 1.0], "dtype": "float32", "layout": "NCHW",
                "resize": None, "pad": None, "finite": True}
COORDINATES46 = {"patch": "strict-divisible", "merge_odd": "right-bottom-zero-pad",
                 "merge_order": ["even-even", "odd-even", "even-odd", "odd-odd"], "token_layout": "BHWC"}
ATTENTION46 = {"qk_scale": "head_dim**-0.5", "relative_index": "signed-2d-row-major",
               "mask_allowed": "0-or-<=-20", "published_blocker": -100.0,
               "shift": "negative-roll-attend-positive-roll"}
LABELS46 = {"0": "vertical", "1": "horizontal", "2": "cross"}
TRAIN_RECIPE46 = {"seed": SEED, "optimizer": "AdamW", "lr": 5e-3, "weight_decay": 1e-4,
                  "steps": 38, "objective": "cross_entropy", "controlled_fixture": True}

published_state46 = {k: v.detach().cpu().clone() for k, v in swin_model.state_dict().items()}
buffer46 = io.BytesIO(); torch.save(published_state46, buffer46)
artifact46 = {
    "artifact_id": "tiny-swin-patterns-v1",
    "config": CONFIG46,
    "state_hex": buffer46.getvalue().hex(),
    "state_digest": state_digest46(published_state46),
    "data_digest": tensor_digest46(train_images46, train_labels46, test_images46, test_labels46),
    "split_recipe": SPLIT46,
    "preprocess": PREPROCESS46,
    "coordinate_recipe": COORDINATES46,
    "attention_recipe": ATTENTION46,
    "labels": LABELS46,
    "training_recipe": TRAIN_RECIPE46,
}
artifact46["package_digest"] = package_digest46(artifact46)
PUBLISHER_REGISTRY46 = MappingProxyType({artifact46["artifact_id"]: artifact46["package_digest"]})

@dataclass(frozen=True)
class PublishedSwin:
    model: TinySwinClassifier
    config: object
    preprocess: object
    coordinates: object
    attention: object
    labels: object
    split: object

    def predict(self, images):
        if images.dtype != torch.float32 or not torch.isfinite(images).all():
            raise ValueError("published Swin expects finite float32 images")
        if images.numel() and (float(images.min()) < 0.0 or float(images.max()) > 1.0):
            raise ValueError("published Swin expects image range [0,1]")
        return self.model(images)

def load_published_swin46(package):
    artifact_id = package.get("artifact_id")
    if artifact_id not in PUBLISHER_REGISTRY46 or package.get("package_digest") != PUBLISHER_REGISTRY46[artifact_id]:
        raise ValueError("artifact is not approved by publisher registry")
    if package_digest46(package) != package["package_digest"]:
        raise ValueError("package metadata digest mismatch")
    expected = {
        "config": CONFIG46, "split_recipe": SPLIT46, "preprocess": PREPROCESS46,
        "coordinate_recipe": COORDINATES46, "attention_recipe": ATTENTION46,
        "labels": LABELS46, "training_recipe": TRAIN_RECIPE46,
    }
    for field, wanted in expected.items():
        if package.get(field) != wanted:
            raise ValueError(f"published semantic contract mismatch: {field}")
    if set(package["labels"]) != {str(i) for i in range(package["config"]["num_classes"])}:
        raise ValueError("label ids and num_classes disagree")
    expected_data = tensor_digest46(train_images46, train_labels46, test_images46, test_labels46)
    if package.get("data_digest") != expected_data:
        raise ValueError("bound split digest mismatch")
    state = torch.load(io.BytesIO(bytes.fromhex(package["state_hex"])), map_location="cpu", weights_only=True)
    if state_digest46(state) != package["state_digest"]:
        raise ValueError("canonical state digest mismatch")
    cfg = package["config"]
    model = TinySwinClassifier(cfg["image_size"], cfg["in_channels"], cfg["num_classes"], cfg["embed_dim"]).eval()
    model.load_state_dict(state, strict=True)
    return PublishedSwin(model, deep_freeze46(cfg), deep_freeze46(package["preprocess"]),
                         deep_freeze46(package["coordinate_recipe"]), deep_freeze46(package["attention_recipe"]),
                         deep_freeze46(package["labels"]), deep_freeze46(package["split_recipe"]))

loaded_swin46 = load_published_swin46(deepcopy(artifact46))
with torch.no_grad():
    assert torch.equal(loaded_swin46.predict(test_images46[:2]), swin_model(test_images46[:2]))
assert loaded_swin46.labels["2"] == "cross"
assert loaded_swin46.coordinates["merge_odd"] == "right-bottom-zero-pad"
try:
    loaded_swin46.labels["0"] = "mutated"
    raise AssertionError("published label map should be read-only")
except TypeError:
    pass

forged46 = deepcopy(artifact46)
forged_state46 = torch.load(io.BytesIO(bytes.fromhex(forged46["state_hex"])), weights_only=True)
forged_state46["head.bias"] = forged_state46["head.bias"] + 7
forged_buffer46 = io.BytesIO(); torch.save(forged_state46, forged_buffer46)
forged46["state_hex"] = forged_buffer46.getvalue().hex()
forged46["state_digest"] = state_digest46(forged_state46)
forged46["labels"] = {"0": "forged", "1": "forged", "2": "forged"}
forged46["package_digest"] = package_digest46(forged46)
try:
    load_published_swin46(forged46)
    raise AssertionError("self-rehashed replacement must fail closed")
except ValueError as exc:
    assert "publisher registry" in str(exc)


## 11. 失败模式、训练/推理差异与生产差距

- **尺寸合同**：patch embedding 拒绝不可整除的输入；patch merging 明确右/下补零。动态 batching 还需把有效 token mask 一路传入 attention。
- **mask 语义**：`-100` 对 float32 足够使概率下溢；混合精度应使用与 dtype 匹配的安全负值并做数值测试。
- **复杂度**：窗口注意力降低 attention 矩阵开销，但 QKV/MLP、feature map 和 window reshape 仍消耗内存；部署要实测峰值 RSS/显存。
- **训练与推理**：本例无 dropout/drop-path；完整复现还要处理 stochastic depth、增强、EMA、AMP 和分布式随机性。
- **发布边界**：`PublishedSwin` 携带只读标签、预处理、坐标与 attention recipe；真实 registry 仍应由签名/KMS/透明日志托管。
- **质量边界**：条纹任务是受控计算图测试，不是自然图像 benchmark。上线需锁定真实 split、校准、漂移监控和回滚制品。

原始资料：

- [Swin Transformer: Hierarchical Vision Transformer using Shifted Windows](https://arxiv.org/abs/2103.14030)
- [PyTorch Conv2d 文档](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)
- [PyTorch LayerNorm 文档](https://pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html)
